# Experiments for Establishing Pro-Cab Lines and Stations

In [ ]:
# imports
import numpy as np
import pandas as pd
from datetime import time
import plotly.graph_objects as go
import ipywidgets as widgets
from ipywidgets import interactive_output, VBox, HBox
from IPython.display import display, clear_output

from operational_vertices import OperationalVertices 
import request_processing_utils
from request_processing_utils import RequestDataframe

In [ ]:
%load_ext autoreload
%autoreload 2

## View Edge and Junction Usage accross an Instance(s)

1. Initialize an RequestDataframe object
2. Add the OSMNX columns that determine graph nodes and edges used for trips with 'add_osmnx_route_columns()'
3. Get the edge and node frequencies with 'get_edge_frequencies()' and 'get_node_frequencies' respectively

In [ ]:
# initialize RDF
# use dir containing multiple json file, a single json file or a previously created RDF stored as .csv file 
OV = OperationalVertices([(51.5558173, 8.4680736), (51.5558173, 8.8597897), (51.7299323, 8.8597897), (51.7299323, 8.4680736)])  # West Paderborn
RDF = RequestDataframe(r'data/input/auto_bf_generation', OV)
# a pre-view of the dataframe in RDF
RDF.df.head()

In [ ]:
# determine the routes with OSMNX and store these values in the dataframe 
RDF.add_osmnx_route_columns()

In [ ]:
edge_freq_df = RDF.get_edge_frequencies()
edge_freq_df.head()

In [ ]:
request_processing_utils.plot_edges_on_map(edge_freq_df.index.to_list(), RDF.G)

In [ ]:
junctions_freq_df = RDF.get_node_frequencies()
junctions_freq_df.head()

In [ ]:
request_processing_utils.plot_nodes_on_map(junctions_freq_df.index.to_list(), RDF.G)

## Plots

Here are plots to view data of the instance files, before calculating any pro-lines, but after estimating routes (durations, distances and coordinates) with OSMNX or GraphHopper.

1. Start (PU) and End Points (DO)
2. Trip Timings
3. Edge usage plot (Not working with OSMNX yet)
4. Node/Junction usage plot

In [ ]:
# initialize RDF
# use dir containing multiple json file, a single json file or a previously created RDF stored as .csv file 
OV = OperationalVertices([(51.5558173, 8.4680736), (51.5558173, 8.8597897), (51.7299323, 8.8597897), (51.7299323, 8.4680736)])  # West Paderborn
RDF = RequestDataframe(r'data/input/auto_bf_generation', OV)
RDF.df.head()

# estimte routes, durations and distances with OSMNX
RDF.add_osmnx_route_columns()
# OR GraphHopper
# RDF.add_graph_hopper_route_columns()

### Locations and Routes on a Map

1. Pick Up, Drop Off Coords and GraphHopper Routes on Map Over Time - Useful to see when which trips are made / what routes are driven when.
2. Single Trips given the index/UserGuid

In [ ]:
# sliders to speify time of day
min_hour_slider = widgets.IntSlider(min=0, max=23, value=0, description="Min Hour", continuous_update=False)
max_hour_slider = widgets.IntSlider(min=0, max=23, value=0, description="Max Hour", continuous_update=False)
min_minute_slider = widgets.IntSlider(min=0, max=59, value=0, step=1, description="Min Minute", continuous_update=True)
max_minute_slider = widgets.IntSlider(min=0, max=59, value=0, step=1, description="Max Minute", continuous_update=True)

# function to convert slider values to times and then to plot the result
def plot_routes_with_sliders(min_hour, min_minutes, max_hour, max_minutes):
    min_time = time(hour=min_hour, minute=min_minutes)
    max_time = time(hour=max_hour, minute=max_minutes)
    RDF.plot_routes_over_time(min_time, max_time, route_mode='lines') 

ui = VBox([
    HBox([min_hour_slider, min_minute_slider]),
    HBox([max_hour_slider, max_minute_slider])
])

out = interactive_output(
    plot_routes_with_sliders,
    {
        'min_hour': min_hour_slider,
        'min_minutes': min_minute_slider,
        'max_hour': max_hour_slider,
        'max_minutes': max_minute_slider
    }
)

display(ui, out)

In [ ]:
# plot single trip
trip_selector_slider = widgets.IntSlider(
    min=0, max=len(RDF.df)-1, value=0, description="UserGuid", continuous_update=False
)
widgets.interactive(RDF.plot_single_trip_route, user_guid=trip_selector_slider)

### Timing Plots

1. Box plots of trip distances and durations
2. Time of Reservations
3. Days of Trips
4. Histogram of Amount Trips Starting Within X Minute Intervals
5. Histogram of Amount Trips Starting/Ending Within 1 Hour Intervals

In [ ]:
RDF.plot_distance_and_duration_boxplots(routing_method="OSMNX")

In [ ]:
RDF.plot_reservation_timings()

In [ ]:
RDF.plot_days_of_trips()

In [ ]:
RDF.plot_trip_timings(interval_mins=30)

In [ ]:
# Extract hour from each datetime column
pickup_start_hours = RDF.df['PickUpStartTime'].dt.hour
pickup_end_hours = RDF.df['PickUpEndTime'].dt.hour
dropoff_start_hours = RDF.df['DropOffStartTime'].dt.hour
dropoff_end_hours = RDF.df['DropOffEndTime'].dt.hour

# Create traces for each histogram
trace1 = go.Histogram(
    x=pickup_start_hours,
    nbinsx=24,
    name='PickUpStartTime',
    opacity=0.6,
    bingroup=1,
    marker=dict(line=dict(width=1, color='black'))
)

trace2 = go.Histogram(
    x=pickup_end_hours,
    nbinsx=24,
    name='PickUpEndTime',
    opacity=0.6,
    bingroup=2,
    marker=dict(line=dict(width=1, color='black'))
)

trace3 = go.Histogram(
    x=dropoff_start_hours,
    nbinsx=24,
    name='DropOffStartTime',
    opacity=0.6,
    bingroup=3,
    marker=dict(line=dict(width=1, color='black'))
)

trace4 = go.Histogram(
    x=dropoff_end_hours,
    nbinsx=24,
    name='DropOffEndTime',
    opacity=0.6,
    bingroup=4,
    marker=dict(line=dict(width=1, color='black'))
)

# Layout settings to ensure all hours are shown
layout = go.Layout(
    barmode='group',  # Overlap the histograms
    bargap=0.5,
    xaxis=dict(
        title='Hour of the Day',
        tickmode='array',
        tickvals=list(range(24)),
        ticktext=[str(i) for i in range(24)],  # 24 hours in the x-axis
        range=[-0.5, 23.5],  # Ensures the x-axis range covers all 24 hours
    ),
    yaxis=dict(title='Count', dtick=5),
    title='Count of Trip Timings Based on the Hour'
)

# Create the figure
fig = go.Figure(data=[trace1, trace2, trace3, trace4], layout=layout)

# Show the plot
fig.show()

### Edge Frequency Plots

Currently not working with OSMNX

In [ ]:
# # sliders to speify time of day
# min_hour_slider = widgets.IntSlider(min=0, max=23, value=0, description="Min Hour", continuous_update=False)
# max_hour_slider = widgets.IntSlider(min=0, max=23, value=0, description="Max Hour", continuous_update=False)
# min_minute_slider = widgets.IntSlider(min=0, max=59, value=0, step=1, description="Min Minute", continuous_update=True)
# max_minute_slider = widgets.IntSlider(min=0, max=59, value=0, step=1, description="Max Minute", continuous_update=True)
# min_freq_slider = widgets.IntSlider(min=1, max=100, value=0, step=1, description="Min Edge Frequency", continuous_update=True)

# def plot_edge_usage_with_sliders(min_hour, min_minutes, max_hour, max_minutes, min_freq):
#     min_time = time(hour=min_hour, minute=min_minutes)
#     max_time = time(hour=max_hour, minute=max_minutes)
#     pu_trace = RDF.get_do_scatter_map(min_time, max_time, size=3, mode='scatter')
#     do_trace = RDF.get_pu_scatter_map(min_time, max_time, size=3, mode='scatter')
#     try:
#         edge_freq_df = RDF.get_edge_frequencies(min_time, max_time)
        
#         # TODO replace this method with to work with OSMNX
#         RND.plot_used_edges(edge_freq_df, min_freq, pu_trace, do_trace)
#     except:
#         print("No Edges to plot")

# ui = VBox([
#     HBox([min_hour_slider, min_minute_slider]),
#     HBox([max_hour_slider, max_minute_slider]),
#     HBox([min_freq_slider])
# ])

# out = interactive_output(
#     plot_edge_usage_with_sliders,
#     {
#         'min_hour': min_hour_slider,
#         'min_minutes': min_minute_slider,
#         'max_hour': max_hour_slider,
#         'max_minutes': max_minute_slider,
#         'min_freq': min_freq_slider
#     }
# )

# display(ui, out)

### Junction Usage Over Time

In [ ]:
# sliders to speify time of day
min_hour_slider = widgets.IntSlider(min=0, max=23, value=0, description="Min Hour", continuous_update=False)
max_hour_slider = widgets.IntSlider(min=0, max=23, value=0, description="Max Hour", continuous_update=False)
min_minute_slider = widgets.IntSlider(min=0, max=59, value=0, step=1, description="Min Minute", continuous_update=True)
max_minute_slider = widgets.IntSlider(min=0, max=59, value=0, step=1, description="Max Minute", continuous_update=True)
min_freq_slider = widgets.IntSlider(min=1, max=len(RDF.df), value=0, step=1, description="Min Junction Frequency", continuous_update=True)

def plot_junctions_with_sliders(min_hour, min_minutes, max_hour, max_minutes, min_freq):
    min_time = time(hour=min_hour, minute=min_minutes)
    max_time = time(hour=max_hour, minute=max_minutes)
    junction_freq_df = RDF.get_node_frequencies(min_time, max_time)
    if len(junctions_freq_df) == 0:
        print("No junctions plot", flush=True)
        return
    request_processing_utils.plot_used_junctions(junction_freq_df, RDF.G, min_freq)
    
ui = VBox([
    HBox([min_hour_slider, min_minute_slider]),
    HBox([max_hour_slider, max_minute_slider]),
    min_freq_slider
])

out = interactive_output(
    plot_junctions_with_sliders,
    {
        'min_hour': min_hour_slider,
        'min_minutes': min_minute_slider,
        'max_hour': max_hour_slider,
        'max_minutes': max_minute_slider,
        'min_freq': min_freq_slider
    }
)

display(ui, out)

## Determine Pro Lines Automatically

### Clustering Method to Generate Lines

1. Cluster PUs and DOs together, by using network constarined DBSCAN clustering on PU and DO locations. This is done with the `determine_lines_with_clustering()` method. PUs and DOs locations of requests are assinged a cluster label. When running determine_lines_with_clustering(), self.df will have 2 columns added called: "PickUpCluster" and "DropOffCluster". These columns contain the cluster labels, to which the PU and DO belong to.

2. With the clusters established, the `determine_cluster_travel_time_and_distances()` method is used to approximate distances and durations between clusters. 
This is useful to because:
    1. Lines should not be created between clusters that are too close (below a desired threshold, currently set by `MINIMUM_CLUSTER_DISTANCE_FOR_LINE`). 
    2. Clusters that are close to each other (below a desired threshold, currently set by `CLOSE_CLUSTER_DISTANCE_THRESHOLD`) can be served by the same line.

3. An OD matrix is contrusted to determine the bidirectional flow of requests between clusters, this is done in `determine_lines_from_cluster_info()` method. In this method PlatoonLine objects created, based on the bidirectional flow between cluster pairs. Lines are created to serve the flow of requests between similar cluster pairs. Similar cluster pairs are those in which each cluster from one pair is within a threshold proximity—based on either geographical distance or travel duration—to a corresponding cluster in the other pair, with the condition that the clusters do not map one-to-one. For example, if the cluster pair (A, B) is similar to (C, D), then A must be close to either C or D, and B must be close to the other. In this method, the `determine_chaining_locations_from_line_info()` method is called to determine the routes (ChainRoute) of a created line and then the Chaining Locations for the lines are determined as well.

4. Finally the requests in the dataframe (self.df of RDF) are assigned values to columns: Line, StartChainingLocation and EndChainingLocation, based on their PickUpCluster and DropOffCluster. If a requests has no line assigned to it, its Line, StartChainingLocation and EndChainingLocation are set to -1.

In [ ]:
# Initialize 
OV = OperationalVertices([(51.5558173, 8.4680736), (51.5558173, 8.8597897), (51.7299323, 8.8597897), (51.7299323, 8.4680736)])  # West Paderborn
RDF = RequestDataframe(r'data/input/auto_bf_generation', OV)
#RDF.df = RDF.df.head()
RDF.df.head()

In [ ]:
# You need to adjust eps and minpts according to how you want cluster to form in the operational area
import os
import pickle

eps = 1000
minpts = 25

pickle_path = f"RDF_{eps}_{minpts}.pkl"
 
if os.path.exists(pickle_path):
    with open(pickle_path, "rb") as f:
        RDF = pickle.load(f)
    print("Loaded object from pickle")
else:
    RDF.determine_lines_with_clustering(eps=eps, minpts=minpts)
    with open(pickle_path, "wb") as f:
        pickle.dump(RDF, f)
    print("Computed and saved object")



##### View Clusters and Cluster Centers

Clusters show to which cluster the PU and DO locations of a request belong to.

Cluster Centers serve as the Origin and Destination for the OD-matrix. And ultimately the start and end locations of lines.

In [ ]:
RDF.plot_pu_and_do_cluster()

In [ ]:
RDF.plot_cluster_centers()

#### View OD Matrix and Duration/Distance Between Clusters

In [ ]:
RDF.plot_trip_frequency_between_clusters(verbose=False)

### Interactive plot to see OD pairs within a time window
- set time window length
- set center of timewindow
- set diagonal elements to zero (for color scheme)
- only trips with PU time window overlap with the chosen window 

In [ ]:


def interactive_od_matrix(obj, time_col="pickup_datetime"):
    start = widgets.DatetimePicker(description="Start", value=None)
    end = widgets.DatetimePicker(description="End", value=None)
    verbose = widgets.Checkbox(description="Verbose", value=False)

    out = widgets.Output()

    def redraw(_=None):
        with out:
            clear_output(wait=True)
            obj.plot_trip_frequency_between_clusters_interactive(
                start_time=start.value,
                end_time=end.value,
                time_col=time_col,
                verbose=verbose.value,
            )

    start.observe(redraw, names="value")
    end.observe(redraw, names="value")
    verbose.observe(redraw, names="value")

    display(widgets.HBox([start, end, verbose]), out)
    redraw()

def interactive_od_time_window(obj, time_col):
    rng = widgets.IntRangeSlider(
        value=(0, 24 * 60 - 1),   # 00:00..23:59
        min=0,
        max=24 * 60 - 1,
        step=1,                  # IMPORTANT
        description="Time:",
        continuous_update=False,
        readout=False,
        layout=widgets.Layout(width="600px"),
    )

    label = widgets.Label()
    out = widgets.Output()

    def minutes_to_hhmm(m):
        h = m // 60
        mm = m % 60
        return f"{h:02d}:{mm:02d}"

    def redraw(_=None):
        start_m, end_m = rng.value
        start_s = minutes_to_hhmm(start_m)
        end_s = minutes_to_hhmm(end_m)
        label.value = f"{start_s} - {end_s}"

        with out:
            clear_output(wait=True)
            obj.plot_trip_frequency_between_clusters_interactive(
                time_col=time_col,
                time_of_day_start=start_s,
                time_of_day_end=end_s,
                verbose=False,
            )

    rng.observe(redraw, names="value")
    display(widgets.VBox([widgets.HBox([rng, label]), out]))
    redraw()

def interactive_od_center_width(obj, time_col):
    center = widgets.IntSlider(value=8*60, min=0, max=24*60-1, step=5,
                               description="Center", continuous_update=False,
                               layout=widgets.Layout(width="800px")
                              )
    width = widgets.IntText(value=120, description="Width (min)",
                            layout=widgets.Layout(width="250px")
                           )
    excl_diag = widgets.Checkbox(value=False, description="Exclude diagonal")
    label = widgets.Label()
    out = widgets.Output()

    def to_hhmm(m):
        h = m // 60
        mm = m % 60
        return f"{h:02d}:{mm:02d}"

    def redraw(_=None):
        w = int(width.value) if width.value is not None else 0
        if w < 0:
            w = 0
        if w > 24*60-1:
            w = 24*60-1

        c = int(center.value)
        half = w // 2
        start_m = c - half
        end_m = c + (w - half)

        if start_m < 0:
            start_m = 0
            end_m = min(24*60-1, start_m + w)
        if end_m > 24*60-1:
            end_m = 24*60-1
            start_m = max(0, end_m - w)

        start_s = to_hhmm(start_m)
        end_s = to_hhmm(end_m)
        label.value = f"Window: {start_s} - {end_s}"

        with out:
            clear_output(wait=True)
            obj.plot_trip_frequency_between_clusters_interactive(
                time_col=time_col,
                time_of_day_start=start_s,
                time_of_day_end=end_s,
                exclude_diagonal=excl_diag.value,
                verbose=False,
            )

    center.observe(redraw, names="value")
    width.observe(redraw, names="value")
    excl_diag.observe(redraw, names="value")

    display(widgets.VBox([center, widgets.HBox([width, excl_diag, label]), out]))
    redraw()
# determine the routes with OSMNX and store these values in the dataframe
# HINT: This also fills the pickup time window in case a dropoff was chosen
#RDF.add_osmnx_route_columns()

# usage:
# interactive_od_matrix(obj, time_col="PickUpEndTime")
#interactive_od_time_window(RDF, time_col="PickUpEndTime")
interactive_od_center_width(RDF, time_col="PickUpEndTime")

### Analyze demand

In [ ]:
bin_len = 30
horizon_start="2025-10-18 05:00:00+02:00"
horizon_end="2025-10-18 23:00:00+02:00"

od_table = RDF.analyze_od_stability(bin_size_min=bin_len, horizon_start=horizon_start, horizon_end=horizon_end)
od_table.head(15)

### Plot bins vs trips 

In [ ]:
request_processing_utils.plot_od_corridor_map(
    od_table,
    bin_size_min=bin_len,
    horizon_start="05:00",
    horizon_end="23:00"
)

### Plot three mostly frequenced ODs bin sizes over time

In [ ]:
RDF.plot_od_time_profile(3, 2, bin_size_min=bin_len)
RDF.plot_od_time_profile(1, 5, bin_size_min=bin_len)
RDF.plot_od_time_profile(0, 2, bin_size_min=bin_len)

### Duration between clusters

In [ ]:
RDF.plot_travel_time_between_clusters()

### View Lines and Select the Lines You Wish to Keep

Lines in the legend of the next plot are ranked in descending order of total requests which are likely to take their assigned line, based on their origin-destinations pairs. 

To see the total number of requests likely to use a line, hover over the Chaining Locations (Larger Makers) of a line.

If you want to get more infomration about a line, like what clusters it is connecting, its estimated duration/distance, etc. use the `.print_line_info()` method. 

In [ ]:
RDF.plot_lines_with_chaining_locations(fixed_cl_size=25)

### Determine Time Table Schedules

1. Select the lines you want to retain, and try to reassign any requests not currently assigned to these lines to one of the selected lines.  
2. View the scheduled start times and estimated arrival times at the start chaining location for trips associated with each line.  
3. There are two methods for determining schedules:
   - Maximum Frequency Approach: `generate_line_schedule_with_max_frequency()`
   - Demand-Driven Approach: `generate_line_schedule()`

In [ ]:
# Select the lines you want to keep, to create the base data
selected_lines = [1, 2, 3, 4]
RDF.reassign_trips_to_lines(selected_lines)  # assign trips to lines that you want to use

In [ ]:
# view when trips assigned to a line will start
RDF.plot_trip_timings_for_lines(line_ids=[1,5])

In [ ]:
# View 
RDF.plot_estimated_arrivals_for_chaining_location(start_chaining_location_id=1)

#### Mehods to Determine Schedules

In [ ]:
# for fixed frequency
x, y = RDF.generate_line_schedule_with_max_frequency(line_id=1)

my_list = [x]
columns = []
for i, obj in enumerate(my_list): 
    d_times = [dt.time() for dt in obj.depart_schedule]
    cl_ids = obj.chaining_schedule
    a_times = [dt.time() for dt in obj.arrival_schedule]
    ucl_ids = obj.unchaining_schedule

    max_len = max(len(d_times), len(cl_ids))
    d_times += [np.nan] * (max_len - len(d_times))
    cl_ids += [np.nan] * (max_len - len(cl_ids))
    a_times += [np.nan] * (max_len - len(a_times))
    ucl_ids += [np.nan] * (max_len - len(ucl_ids))

    # Create a DataFrame with MultiIndex columns
    df = pd.DataFrame({
        (f"Pro {i+1} Schedule", "DTime"): d_times,
        (f"Pro {i+1} Schedule", "CL"): cl_ids,
        (f"Pro {i+1} Schedule", "ATime"): a_times,
        (f"Pro {i+1} Schedule", "UCL"): ucl_ids,
        (f"Pro {i+1} Schedule", " "): "    "
    })
    
    columns.append(df)
combined_df = pd.concat(columns, axis=1)

print("Total trips:", len(combined_df))
print("max freq:", y//60, "minutes")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)
print("\nSCHEDULE:")
combined_df

In [ ]:
# demand driven approach
platoon_time_tables = RDF.generate_line_schedule(line_id=1, avg_cutoff=0.0001, available_pros=1)
print(f"Total trips: {len(platoon_time_tables[0].depart_schedule)}")
print(f"{len(platoon_time_tables)} Pros used to serve this Line")
columns = []

for i, obj in enumerate(platoon_time_tables): 
    d_times = [dt.time() for dt in obj.depart_schedule]
    cl_ids = obj.chaining_schedule
    a_times = [dt.time() for dt in obj.arrival_schedule]
    ucl_ids = obj.unchaining_schedule

    max_len = max(len(d_times), len(cl_ids))
    d_times += [np.nan] * (max_len - len(d_times))
    cl_ids += [np.nan] * (max_len - len(cl_ids))
    a_times += [np.nan] * (max_len - len(a_times))
    ucl_ids += [np.nan] * (max_len - len(ucl_ids))

    # Create a DataFrame with MultiIndex columns
    df = pd.DataFrame({
        (f"Pro {i+1} Schedule", "DTime"): d_times,
        (f"Pro {i+1} Schedule", "CL"): cl_ids,
        (f"Pro {i+1} Schedule", "ATime"): a_times,
        (f"Pro {i+1} Schedule", "UCL"): ucl_ids,
        (f"Pro {i+1} Schedule", " "): "    "
    })
    
    columns.append(df)
combined_df = pd.concat(columns, axis=1)
print("\nSCHEDULE:")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)
combined_df.fillna("")

### Generate Base Data

Generate a base data file with the selected lines. The 'max frequency schedules' are generated in `generate_ini_base_data()`.

In [ ]:
# RDF.generate_ini_base_data(fpath='TEST_FILE.json', line_ids=[1])
RDF.generate_ini_base_data(
    fpath="TEST_FILE.json",
    line_ids=selected_lines,
    service_start="05:00",
    service_end="23:00",
    # pros_per_line=[2,1]
)

## Generate Pro Lines Given Desired Start and End Locations

1. Initailize `RequestDataframe` with instance files and a `OperationalVertices`
2. Add Lines with `add_line_between_coords()`
3. Assign the resquets to lines with `reassign_trips_to_lines()`
4. Generate base data

In [ ]:
## Initialize ReuquestDataframe with instances
OV = OperationalVertices([(51.5558173, 8.4680736), (51.5558173, 8.8597897), (51.7299323, 8.8597897), (51.7299323, 8.4680736)])  # West Paderborn 
RDF = RequestDataframe(r'data/input/auto_bf_generation', OV)
RDF.df.head()

In [ ]:
# Create Lines between disired coordinates
# Line for OVENHAUSEN to Hoexter VIA LUETMARSEN and BREMKHAUSEN
RDF.add_line_between_coords(start=(9.298855, 51.777650), 
                            end=(9.378526, 51.779135), 
                            via=[(9.342257, 51.784791), (9.356289, 51.796902)])

# Line for Bosseborn to Hoexter
RDF.add_line_between_coords(start=(9.305268, 51.750597), 
                            end=(9.374094, 51.774026), 
                            via=None)

In [ ]:
RDF.plot_lines_with_chaining_locations(fixed_cl_size=25)

In [ ]:
RDF.reassign_trips_to_lines(line_ids=[1,2], min_distance=5000)

In [ ]:
RDF.generate_ini_base_data(fpath=r"TEST_FILE.json", line_ids=[1,2], pros_per_line=[1,1])